### Group & Responsibilities

**Group Number:** 7

| Member | Student ID | Contributions |
|---|---|---|
| ZHANG SHIHAN | A0330526M | Responsible for the Streamlit interface and reasoning trace presentation; application deployment; overall testing of all five Sudoku puzzles; FC/BC runtime comparison; Notebook organization and final submission checks; and Part B Questions 4 and 5. |
| FANG XUYI | A0353836W | Responsible for project coordination, encoding the general propositional and Horn knowledge bases, the backward-chaining algorithm and full-grid Sudoku solving with backward chaining, algorithm correctness and cyclic-dependency testing, and Part B Question 3. |
| CHEN JUNXIAO | A0356288M | Responsible for Notebook and code integration; full-grid Sudoku solving with forward chaining; and Part B Questions 1 and 2, explaining the general and Horn knowledge-base representations and analysing the computational cost of model checking and resolution. |
| CHOOG SHENG HONG | Pending confirmation | Responsible for reviewing all parts of the assignment and refining the final submission. |


# Sudoku Knowledge Representation & Inference

You will implement three functions -- `build_general_kb`, `build_definite_kb`, `pl_bc_entails` -- and the full-grid solving logic in `sudoku_solver.py`.

This notebook imports and tests those functions. The Streamlit app (`sudoku_app.py`) must import the same implementation from `sudoku_solver.py`; do not copy or rewrite the solver functions inside the app.

**Rules:**

- In `sudoku_solver.py`, import only from `utils.py` and `logic_.py`; do not modify either file.
- Do not duplicate the core solver functions in this notebook or `sudoku_app.py`.
- All other content in this notebook may be edited freely.


In [1]:
from utils import *
from logic_ import *
import json
import time
import importlib
import sudoku_solver

# Reload so edits made to sudoku_solver.py are picked up when this cell is rerun.
importlib.reload(sudoku_solver)

from sudoku_solver import (
    atom,
    build_general_kb,
    build_definite_kb,
    solve_full_grid_fc,
    pl_bc_entails,
    solve_full_grid_bc,
)


## Loading a puzzle from JSON

Puzzles are provided as JSON, not embedded in this notebook. Each file looks like:

```json
{
  "n": 9, "box_h": 3, "box_w": 3,
  "puzzles": [
    {
      "givens": {"1_1": 3, "2_3": 1, ...},
      "given_count": 28,
      "solution": {"1_1": 3, "1_2": 4, ...}
    },
    ...
  ]
}
```

`"r_c"` string keys map to the value at row `r`, column `c` (1-indexed). `given_count` is exactly how many cells are given. `solution` is included so you can check your own work as you go, but your functions must not read `solution` to answer a query, they should only read `givens`.

In [2]:
#do not change this function, it is used to load the puzzle pool from a json file
def load_pool(path):
    with open(path) as f:
        raw = json.load(f)
    puzzles = []
    for p in raw['puzzles']:
        givens = {tuple(int(x) for x in k.split('_')): v for k, v in p['givens'].items()}
        solution = {tuple(int(x) for x in k.split('_')): v for k, v in p['solution'].items()}
        puzzles.append({'givens': givens, 'solution': solution, 'given_count': p['given_count']})
    return raw['n'], raw['box_h'], raw['box_w'], puzzles

n, box_h, box_w, puzzle_pool = load_pool('puzzles.json')
print(f'{len(puzzle_pool)} puzzles loaded, {n}x{n} grid, {box_h}x{box_w} boxes')
print('given_count values:', sorted(p['given_count'] for p in puzzle_pool))

# Pick one puzzle to work with through the rest of this notebook.
puzzle = puzzle_pool[0]
givens = puzzle['givens']
print(f"working puzzle has {puzzle['given_count']} givens")
for r in range(1, n + 1):
    print([givens.get((r, c), ' ') for c in range(1, n + 1)])

5 puzzles loaded, 9x9 grid, 3x3 boxes
given_count values: [30, 33, 36, 39, 42]
working puzzle has 30 givens
[' ', 3, ' ', ' ', ' ', ' ', ' ', ' ', ' ']
[2, ' ', ' ', ' ', ' ', 7, 8, 6, ' ']
[5, 8, ' ', 2, 6, ' ', ' ', 3, ' ']
[7, 5, ' ', ' ', ' ', ' ', ' ', 8, ' ']
[' ', ' ', ' ', ' ', 7, ' ', 5, ' ', 4]
[' ', ' ', ' ', 5, 3, ' ', ' ', 9, 6]
[' ', 1, 2, ' ', ' ', 9, ' ', ' ', ' ']
[6, 4, ' ', ' ', 5, 8, 9, ' ', ' ']
[' ', ' ', ' ', ' ', 2, 3, ' ', ' ', ' ']


Notice: `pl_fc_entails`/`pl_resolution`/`tt_entails` are all given to you, already implemented, in `logic_.py`. The one algorithm you write yourself in this assignment is **backward chaining** (`pl_bc_entails`) -- see Task 2.

## Define Symbols

Two families of propositional symbols, for row $r$, column $c$, value $v$ (all ranging over $1$ to $n$):

| Symbol | Meaning |
|---|---|
| $\mathit{Is}_{rcv}$ | cell $(r,c)$ has value $v$ |
| $\mathit{Not}_{rcv}$ | cell $(r,c)$ does **not** have value $v$ |

For each representation in Task 1, determine which of these two families is actually needed.

Hint: consider what form a definite (Horn) clause must take, and what `PropDefiniteKB.tell()` will accept.

### Helper function

`atom(prefix, r, c, v)`, where prefix can be either `Is` or `Not`, is a naming helper so propositional symbols need not be typed. It is provided for you in `sudoku_solver.py`; do not change it.

In code, $\mathit{Is}_{rcv}$ and $\mathit{Not}_{rcv}$ are written as single-word symbol names, e.g. `Is3_2_4` and `Not3_2_4`: the prefix is followed immediately by `r`, then by `c` and `v` separated by underscores. No separator is needed between the prefix and `r` since `expr()` only requires a symbol name to start with an uppercase letter. So `Is3_2_4` is parsed as one valid symbol.


In [3]:
# atom() is provided in sudoku_solver.py and imported above.


## Part A: Design the Sudoku Solver

### A.1) Knowledge Representation - Build KB

Every well-posed Sudoku puzzle satisfies exactly these conditions:

- Each cell is assigned **at least one** value from $\{1, \dots, n\}$.
- Each cell is assigned **at most one** value from $\{1, \dots, n\}$, i.e., it cannot hold two different values at once.
- No two cells in the same row hold the same value.
- No two cells in the same column hold the same value.
- No two cells in the same box hold the same value.
- The **givens** cells hold their stated values.

Formalize these Sudoku constraints as propositional logic, in **two** representations:

**(a) General clauses -  `build_general_kb`.**

Encode each of the following directly: no restriction here; you may use arbitrary disjunctions of positive or negated literals. Must return a `PropKB`. 

**(b) Definite (Horn) clauses:- `build_definite_kb`.** Must return a `PropDefiniteKB`. Recall a definite clause is a disjunction of literals with exactly one *positive* literal.

Equivalently written as an implication whose conclusion is a single positive literal and whose premises are a conjunction of positive literals: `P1 & P2 & ... & Pk ==> Q`.

`PropDefiniteKB.tell()` will reject anything else.

Both functions take the puzzle's givens as fixed facts.

- **Implement `build_general_kb()` and `build_definite_kb()` in sudoku_solver.py**
- **Rerun the import cell near the top of this notebook after making changes.**
- **Explain your representation in Conceptual Question 1 below.**

In [4]:
# Implement build_general_kb() and build_definite_kb() in sudoku_solver.py.
# Rerun the import cell near the top of this notebook after making changes.


### A.2) Solve the Puzzle

**(a) Resolution and model checking on the general representation.** Using `build_general_kb` and the library's `pl_resolution` / `tt_entails`, try to solve the puzzle, i.e., for each cell, determine which value is entailed. Attempt this in the code cell below: it is provided commented out, because both are sound and complete on `build_general_kb`'s output but neither scales to the full grid, and it is expected to hang or take an impractically long time. Uncomment a few lines at a time and give each at most about 30 seconds; use Kernel > Interrupt if it hasn't returned by then, and note what you observed.

Explain in your own words: what specifically makes `pl_resolution`'s cost grow out of control here, and separately, what makes `tt_entails`'s cost grow out of control? A simple complexity argument for each is the expected answer.

**Use your observations in Conceptual Question 2 below.**


In [5]:
# Attempt (a): try solving the puzzle using pl_resolution / tt_entails on the
# general KB. Left commented out because it is expected to take an
# impractically long time (pl_resolution) or be outright infeasible
# (tt_entails) on the full grid.
#
# Uncomment a few lines at a time and give each at most ~30 seconds; use
# Kernel > Interrupt if it hasn't returned by then.
#
general_kb = build_general_kb(n, box_h, box_w, givens)
r, c, v = 1, 1, givens.get((1, 1), 1)  # try one cell/value pair
query = atom('Is', r, c, v)

# print(pl_resolution(general_kb, query))    
# print(tt_entails(associate('&', general_kb.clauses), query)) 

### Observation notes

Record what you observed from the resolution/model-checking experiment here. Use these observations when answering Conceptual Question 2 below.


### Observation: 
We tried the query `Is1_1_1` with `pl_resolution` and `tt_entails` separately. Neither returned a `True` or `False` result within minutes, so we interrupted them. 
we commented out both calls again so the notebook can run from start to finish. These interrupted attempts show that the methods were impractical for this query in our test.

**(b) Forward chaining on the full grid --** Implement `solve_full_grid_fc()` in sudoku_solver.py. Using your above `build_definite_kb` and the library's `pl_fc_entails` (no need to reimplement them), solve the whole puzzle. Find the value that's entailed for every cell. Reconstruct and display the solved grid.

**(c) Backward chaining -- implement it yourself.**
```python
pl_bc_entails(kb, query) -> bool
```
Implement `pl_bc_entails()` in sudoku_solver.py. Start from the query and recursively try to prove each premise of a rule whose conclusion matches the current goal, bottoming out at known facts. Your function must agree with `pl_fc_entails` on every cell/value pair in this puzzle including correctly returning `False` for values that are *not* part of the solution.

**(d) Backward chaining on the full grid --** Implement `solve_full_grid_bc` in sudoku_solver.py. Using your above `build_definite_kb` and your own `pl_bc_entails`, solve the whole puzzle the same way `solve_full_grid_fc` does: for every cell, try each candidate value until `pl_bc_entails` confirms one. Time both `solve_full_grid_fc` and `solve_full_grid_bc` on the same puzzle and compare. Use your measured result where relevant in Conceptual Question 5.


In [6]:
# Implement solve_full_grid_fc(), pl_bc_entails(), and solve_full_grid_bc()
# in sudoku_solver.py. Rerun the import cell near the top after making changes.


### Verifying the algorithms

Uncomment the following block of code to validate your code.

In [7]:
# Verify solve_full_grid_fc against the puzzle's known solution. Then check
# pl_bc_entails directly, for completeness (it finds the correct value) and
# soundness (it never wrongly confirms an incorrect one), before timing
# solve_full_grid_bc against the same solution.
#
import time

t0 = time.time()
solved = solve_full_grid_fc(n, box_h, box_w, givens)
fc_time = time.time() - t0
assert solved == puzzle['solution']


definite_kb = build_definite_kb(n, box_h, box_w, givens)

# Completeness: pl_bc_entails must find the correct value for every cell.
for (r, c), v in puzzle['solution'].items():
    assert pl_bc_entails(definite_kb, atom('Is', r, c, v)) == True

# Soundness: pl_bc_entails must not also confirm any incorrect value.
for (r, c), v in puzzle['solution'].items():
    for other_v in range(1, n + 1):
        if other_v != v:
            assert pl_bc_entails(definite_kb, atom('Is', r, c, other_v)) == False

t0 = time.time()
solved_bc = solve_full_grid_bc(n, box_h, box_w, givens)
bc_time = time.time() - t0
assert solved_bc == puzzle['solution']

print(f"solve_full_grid_fc: {fc_time:.2f}s")
for r in range(1,n+1):
    for c in range(1,n+1):
        print(solved.get((r,c)),end=' ')
    print('')
print(f"solve_full_grid_bc: {bc_time:.2f}s")
for r in range(1,n+1):
    for c in range(1,n+1):
        print(solved_bc.get((r,c)),end=' ')
    print('')

solve_full_grid_fc: 15.12s
1 3 6 9 8 5 4 7 2 
2 9 4 3 1 7 8 6 5 
5 8 7 2 6 4 1 3 9 
7 5 1 4 9 6 2 8 3 
3 6 9 8 7 2 5 1 4 
4 2 8 5 3 1 7 9 6 
8 1 2 6 4 9 3 5 7 
6 4 3 7 5 8 9 2 1 
9 7 5 1 2 3 6 4 8 
solve_full_grid_bc: 0.76s
1 3 6 9 8 5 4 7 2 
2 9 4 3 1 7 8 6 5 
5 8 7 2 6 4 1 3 9 
7 5 1 4 9 6 2 8 3 
3 6 9 8 7 2 5 1 4 
4 2 8 5 3 1 7 9 6 
8 1 2 6 4 9 3 5 7 
6 4 3 7 5 8 9 2 1 
9 7 5 1 2 3 6 4 8 


In [8]:
# Validate the remaining provided puzzles with the same imported solver functions.
for index, entry in enumerate(puzzle_pool[1:], 2):
    start = time.perf_counter()
    fc_grid = solve_full_grid_fc(n, box_h, box_w, entry['givens'])
    fc_elapsed = time.perf_counter() - start
    start = time.perf_counter()
    bc_grid = solve_full_grid_bc(n, box_h, box_w, entry['givens'])
    bc_elapsed = time.perf_counter() - start
    assert fc_grid == bc_grid == entry['solution']
    assert all(fc_grid[cell] == value for cell, value in entry['givens'].items())
    kb = build_definite_kb(n, box_h, box_w, entry['givens'])
    for (r, c), correct in entry['solution'].items():
        for value in range(1, n + 1):
            assert pl_bc_entails(kb, atom('Is', r, c, value)) == (value == correct)
    print(f'Puzzle {index}: FC={fc_elapsed:.3f}s; BC={bc_elapsed:.3f}s; '
          'both full grids and 729 BC candidate queries PASS')
print('All five puzzles: both full grids and 3,645 BC candidate queries PASS')


Puzzle 2: FC=23.347s; BC=0.737s; both full grids and 729 BC candidate queries PASS


Puzzle 3: FC=19.162s; BC=0.440s; both full grids and 729 BC candidate queries PASS


Puzzle 4: FC=11.197s; BC=0.482s; both full grids and 729 BC candidate queries PASS


Puzzle 5: FC=11.558s; BC=0.381s; both full grids and 729 BC candidate queries PASS
All five puzzles: both full grids and 3,645 BC candidate queries PASS


## Part B: Conceptual Questions

Answer all five questions directly in this notebook. Replace each **Your answer:** placeholder with your own response.


### 1. Detailed Representation Strategy: General vs. Definite (Horn) Encoding

Explain in detail how you formalized the Sudoku puzzle constraints into propositional logic across both Knowledge Base representations:

**(a) General KB Strategy (`build_general_kb`):** Detail how standard Sudoku rules (e.g., at-least-one value per cell, at-most-one value per cell, row/column/box uniqueness) are directly translated into Conjunctive Normal Form (CNF) clauses without structural restrictions.

**(b) Definite KB Strategy (`build_definite_kb`):** Definite/Horn clauses strictly permit at most one positive literal per clause, prohibiting disjunctive constraints like $(Is_{r,c,1} \lor Is_{r,c,2} \lor Is_{r,c,3} \lor ... \lor Is_{r,c,n})$. Explain step-by-step how your encoding deals with this issue.


**Your answer:**

#### **(a) General KB Strategy**


##### (1) At-least-one value per cell

For every cell $(r,c)$, we add:

$$
Is_{r,c,1}\lor Is_{r,c,2}\lor\cdots\lor Is_{r,c,n}.
$$

This clause requires at least one candidate value to be true. In the implementation, the outer two loops iterate over every row and column. For each cell, `clause = atom('Is', r, c, 1)` starts with the atom for value 1. The inner loop considers values 2 through (n). The statement `clause |= atom('Is', r, c, v)` extends the expression repeatedly using `|` to add the remaining values.


##### (2) At-most-one value per cell

For every cell and every pair of different values $v<w$, we add:

$$
\neg Is_{r,c,v}\lor\neg Is_{r,c,w}.
$$

This is equivalent to saying that the cell cannot contain both values simultaneously. Starting the second value at v + 1 ensures that each unordered pair of values is considered only once. Together with the at-least-one clause, these clauses require exactly one value per cell.


##### (3)row/column/box uniqueness

For every pair of distinct cells $(r,c)$ and $(r',c')$ sharing a row, column, or box, and for every value $v$, we add:

$$
\neg Is_{r,c,v}\lor\neg Is_{r',c',v}.
$$

Thus, two cells in the same Sudoku unit cannot contain the same value.

Our implementation combines these three types of uniqueness constraints in one cell-pair loop. Particularly, two cells belong to the same box precisely when:

$$
\left\lfloor\frac{r-1}{\text{box\_h}}\right\rfloor
=
\left\lfloor\frac{r'-1}{\text{box\_h}}\right\rfloor
$$

and

$$
\left\lfloor\frac{c-1}{\text{box\_w}}\right\rfloor
=
\left\lfloor\frac{c'-1}{\text{box\_w}}\right\rfloor.
$$

The loop skips the current cell and previously considered pairs. Since the condition uses “same row OR same column OR same box,” a pair sharing more than one unit still receives only one exclusion clause per value.


##### (4)Given values

For every given entry $(r,c)=v$, we add the positive unit clause:

$$
Is_{r,c,v}.
$$

These facts fix the initial values of the puzzle. The function reads only `givens`, not the supplied solution.

All these expressions are already CNF clauses: the at-least-one constraints are disjunctions of positive literals, the exclusion constraints are disjunctions of two negative literals, and the givens are unit clauses. We add them using `kb.tell()`. The entire knowledge base is the conjunction of these clauses. Although `PropKB` permits general propositional expressions, our encoding directly supplies clauses in CNF.


#### **(b) Definite KB Strategy**


Our second implementation creates a `PropDefiniteKB`. A definite clause has exactly one positive literal and can be written as:

$$
P_1\land P_2\land\cdots\land P_k\Rightarrow Q,
$$

where the premises and conclusion are positive atoms. Positive facts are also permitted.

##### (1) At-least/most-one value per cell

We use two families of atoms:

- $Is_{r,c,v}$: cell $(r,c)$ has value $v$.
- $Not_{r,c,v}$: value $v$ has been ruled out for cell $(r,c)$.

Crucially, $Not_{r,c,v}$ is an independent positive atom, not the expression $\neg Is_{r,c,v}$. 

For every cell and every pair of different values $v\ne w$, we add:

$$
Is_{r,c,v}\Rightarrow Not_{r,c,w}.
$$

Once a cell is known to contain $v$, every other value is eliminated from that cell.

Unlike the unordered pairs in the general encoding, both directions are needed here. For example, “value 3 eliminates value 5” and “value 5 eliminates value 3” are different implications.

##### (2)row/column/box uniqueness

For every ordered pair of distinct cells sharing a row, column, or box (the way to check whether two cells belong to the same box is the same as that in General KB), and every value $v$, we add:

$$
Is_{r,c,v}\Rightarrow Not_{r',c',v}.
$$

A confirmed value therefore eliminates the same value from all other cells in the corresponding units. Our loops consider both directions between two cells and exclude self-pairs.

##### (3)Infer the last remaining candidate

For every cell and every candidate value $v$, we add:

$$
\left(
\bigwedge_{\substack{w=1\\w\ne v}}^{n}
Not_{r,c,w}
\right)
\Rightarrow Is_{r,c,v}.
$$

If all other candidates have been eliminated, the remaining value is the right one to fill. 

The implementation builds the conjunction of elimination atoms using `&`, then constructs the implication using `|'==>'|`.


##### (4)Given values

For every given entry $(r,c)=v$, we add the positive unit clause:

$$
Is_{r,c,v}.
$$

These facts fix the initial values of the puzzle. The function reads only `givens`, not the supplied solution.

The last-candidate rule uses Sudoku's requirement that every cell has a value,
but does not assert the non-Horn disjunction in the definite KB. These rules
encode sound elimination consequences, not a logically equivalent replacement
for the full CNF constraints over independent Is and Not atoms. For example, an
empty definite KB has a least model with no Is values established, whereas the
CNF requires a value in every cell. Failure to derive an Is atom is not evidence
that its corresponding Not atom is true. The implemented rule set need not
solve every Sudoku; it is tested on the supplied puzzles without guessing.


### 2. Theoretical Completeness vs. Computational Tractability
Model checking and resolution-refutation are sound and complete—they are guaranteed to terminate with a correct answer for any propositional KB. Despite this guarantee, explain whether you would use either as the default algorithm for solving Sudoku puzzles. *(Hint: Consider space/time complexity and state-space growth, and use your observations from the experiment above where relevant.)*


**Your answer:**

We would not use either model checking or resolution-refutation as the default algorithm for solving Sudoku puzzles. In our experiment above on `Is1_1_1`, neither returned a Boolean result before it was interrupted. Although the two algorithm are sound and complete to guarantee correct inference when the algorithms finish, but they do not guarantee practical running time or memory usage.

**Model checking.** For `tt_entails`, the main problem is the exponential number of Boolean assignments. It evaluates the KB after all symbols have been assigned; it does not prune partial assignments using Sudoku constraints. Our encoding uses $n^3$ atoms for an $n\times n$ Sudoku, giving $2^{n^3}$ possible assignments. For a 9×9 grid, this means 729 atoms and $2^{729}\approx2.8\times10^{219}$ assignments. These assignments are not necessarily valid Sudoku solutions. The truth-table procedure may explore all of them in the worst case, with additional work to evaluate the formulas. Recursive enumeration avoids storing the entire truth table at once, but the time requirement remains prohibitive.


**Resolution-refutation.** For `pl_resolution`, the main problem is the growth of the clause set and the number of clause pairs considered. With $N$ clauses, there are $N(N-1)/2$ distinct pairs in a round. In our experiment on `Is1_1_1`, our general KB contained 10,317 clauses, corresponding to approximately 53.2 million pairs even before adding the negated query. Not every pair produces a useful resolvent, but examining pairs still incurs substantial overhead. Newly derived clauses increase both subsequent pairing work and memory consumption. The supplied code constructs the list before resolving pairs, giving quadratic work and memory in the current clause count. Later rounds can add many resolvents and repeat this work; the set of possible propositional clauses grows exponentially with the number of atoms.

Our experiment illustrates this difference between theoretical guarantees and practical performance. `build_general_kb` generated 10,317 clauses in approximately 0.2 seconds, but neither `pl_resolution` nor `tt_entails` returned a result for the single query `Is1_1_1` during our attempts, so we interrupted both. This shows that even one query was impractical in our test.
w


### 3. Backward Chaining: Design, Pseudocode, and Challenges

Write pseudocode for `pl_bc_entails(kb, query)`, the backward-chaining algorithm you implemented. Show, at a level of detail that reveals the algorithm's structure (not full Python), how the function checks whether the query is already a known fact, finds candidate rules whose conclusion matches the current goal, recursively proves each premise of such a rule, and combines results—both across the premises of one rule and across multiple candidate rules—to reach a single boolean answer.

Then, in your own words, discuss the design challenges you had to work through to make your algorithm both correct and guaranteed to terminate on every puzzle, and explain how your pseudocode addresses them.


**Your answer:**

In [4]:
# PL_BC_ENTAILS(kb, query):

#     If the KB has no inference records yet, prepare:
#         facts: facts supplied directly by the puzzle
#         rules: rule premises indexed by their conclusion
#         proven: propositions successfully proved later
#         expanded: goals whose candidate rules have already been examined
#         waiting: rules indexed by an unproved premise they need
#         remaining: the number of unproved premises left for each rule

#     Create deferred: goals postponed during this query because of depth

#     MARK_PROVEN(symbol):
#         Put symbol in a worklist
#         While the worklist is not empty:
#             Take one proposition from the worklist
#             If it is already a fact or proven, continue
#             Add it to proven
#             For each rule waiting for it:
#                 Decrease remaining for that rule by one
#                 If remaining becomes zero:
#                     Add the rule's conclusion to the worklist

#     PROVE(q, depth):
#         If q is in facts or proven, succeed
#         If q is in expanded, stop re-expanding it; report "not yet proven"
#         If the depth limit is reached:
#             Add q to deferred
#             Report "not yet proven"

#         Add q to expanded

#         For each rule whose conclusion is q:      
#             Find the premises not yet proven
#             Record their number in remaining

#             If no unproved premises remain:
#                 MARK_PROVEN(q)
#                 Succeed

#             Register in waiting which premises this rule needs
#             Recursively call PROVE for those premises  # AND within a rule

#             If q has consequently entered proven, succeed

#         Report "not yet proven"

#     Call PROVE(query, 0)
#     Process each goal in deferred by calling PROVE again

#     Return True if query is in facts or proven; otherwise return False

### Design Challenges

#### · Deciding what state to store and where to store it

To improve efficiency and avoid repeated checks, 「facts」 for the given facts and 「rules」 indexed by conclusion should be stored on the KB object; the KB should also store 「proven」 for propositions proved—these three are relatively easy to think of.

The difficult part is storing and managing propositions whose rules have begun to be explored but which have not yet been proven, so that a cycle does not repeatedly restart the same search. 「expanded」 records these goals. It prevents repeated expansion, but an expanded goal is not treated as false.

#### · Updating dependent rules when a proposition is proven

「waiting」 records which rules depend on each unproved premise, while 「remaining」 records how many premises each rule still needs.

「mark_proven」 marks a proposition as proven and updates these records. When it establishes a proposition, it updates the rules waiting for it. If a rule needs no further premises, its conclusion enters 「proven」, even if that conclusion was already in 「expanded」.

#### · Handling excessive recursion depth

A long chain of subgoals may exceed Python’s recursion limit. When 「prove」 reaches the depth limit, the goal is placed in 「deferred」 rather than treated as false. After the initial search, the deferred goals are processed again from depth zero. Since each goal is expanded at most once, cycles cannot repeatedly expand the same finite set of goals. Only after the deferred goals have been processed does the function return a final `False` if the query is in neither 「facts」 nor 「proven」. This means the query was not derived from the KB, not that its negation was proved.

#### · AND/OR, termination, and implementation scope

All unproved premises of one rule must become established before its remaining
counter reaches zero (AND). Any candidate rule for the goal can establish that
goal (OR); a cyclic route does not prevent trying another rule with a grounded
proof. A cycle without a fact cannot create a proof. Successful conclusions are
propagated with an iterative worklist, including to goals already expanded.
Thus an intermediate failure is not permanently cached as a false proposition.
For a finite puzzle KB, each goal is expanded at most once, each newly proved
atom is propagated once, and deferred goals restart at depth zero. This bounds
recursive depth and makes processing terminate.

This argument applies to the generated Sudoku KB: premises within each rule are
distinct, and the KB is not changed after inference begins. The current code
counts premise occurrences but stores waiting rules in sets, so it does not
handle repeated premises in an arbitrary Horn rule reliably. It also caches
facts and rules on the KB without detecting later tell/retract changes. Our
Sudoku solvers and the app build fresh KBs from givens and do not mutate their
clauses during inference. No cache-invalidation or duplicate-premise repair is
claimed for this submitted implementation.


### 4. Expressive Limits of Horn Logic

Named elimination techniques such as Naked Pairs and X-Wing can, in fact, be encoded as definite clauses, using the same `Is`/`Not` vocabulary as your `build_definite_kb`. Work out how you would encode one of these techniques as definite clauses, and discuss the consequences of doing so.


**Your answer:**

We use **Naked Pairs**. Here $N_{av}$ denotes the positive atom `Not` for value $v$ in cell $a$. Consider distinct cells $a,b$ in the same unit (row, column,
or box), and values $p\ne q$. Suppose every other value has been explicitly excluded
from both cells. Define the positive conjunction

$$D_{a,b,p,q}=\bigwedge_{v\notin\{p,q\}}(N_{av}\land N_{bv}).$$

For each other cell $c$ in that unit, add two **separate** definite rules:

$$D_{a,b,p,q}\Rightarrow N_{cp},\qquad D_{a,b,p,q}\Rightarrow N_{cq}.$$

For example, let $a=(1,1)$, $b=(1,2)$, and $\{p,q\}=\{2,7\}$. The body is the
conjunction of the 14 positive Not atoms excluding values $1,3,4,5,6,8,9$ from
each of those two cells. It implies `Not1_3_2` in one rule and `Not1_3_7` in another,
and similarly for columns 4–9 in row 1. Each rule has exactly one positive head.
The body is written out using the existing atoms; $D$ above is only notation,
not a required new proposition.

Why this is sound: in a valid Sudoku the two cells each need a value, both values
come from $\{p,q\}$, and the cells cannot have equal values because they share a
unit. They must therefore use $p$ and $q$ in some order. Neither value can appear
in another cell of that unit. It is enough that their candidate sets are subsets
of this pair; we do not infer unproven candidates by treating missing Not facts as
false, and we do not guess which cell takes which value.

These rules can trigger extra eliminations when last-candidate reasoning alone
stalls. The cost is a much larger KB: a naive 9×9 grounding considers 27 units,
$\binom92$ pairs of cells, $\binom92$ pairs of values, 7 other cells, and 2 heads:
$27\times36\times36\times7\times2=489{,}888$ candidate rules before deduplication.
Each body has 14 premises. This increases construction time, memory, and premise
processing. Rule indexing or generating relevant rules only when needed can help.
Even this extension would not make the rule set sufficient for every Sudoku.
The submitted solver intentionally implements the required elimination and
last-candidate rules; this question develops the extension without silently
changing the measured baseline.

### 5. Data-Driven vs. Goal-Driven Performance

Forward chaining (data-driven) and backward chaining (goal-driven) are both sound and complete for Horn KBs, but their execution runtimes vary depending on the target query.

**(a)** Describe a scenario—in terms of total KB size versus the query-relevant subset—where backward chaining is significantly faster than forward chaining.

**(b)** Describe a scenario where backward chaining offers no performance advantage, or performs worse than forward chaining.

Use your measured forward- vs. backward-chaining result where relevant.


**Your answer:**

**(a)** Backward chaining can be faster when a large KB contains many rules
unrelated to the requested atom, while the query has a short proof using a small
relevant subset. Indexing rules by conclusion avoids exploring unrelated rule
bodies after that index is built. In this implementation the first query still
scans the whole KB to construct its rule/fact records, so this initial cost must
not be confused with proof-search cost. Later queries on the same unchanged KB
reuse those records and established proofs.

**(b)** If queries cover most conclusions, or a query depends on almost the whole
KB, goal-directed exploration offers little pruning. A forward closure computed
once can then serve many queries efficiently. Backward chaining without proof
reuse can repeat work; maintaining goal, waiting-rule and remaining-premise
records also takes time and space.

The timing cells above measure the two imported full-grid solvers on the same
puzzles using fresh KBs. Both timings include KB construction; FC additionally
builds its premise index, and BC builds its inference records on the first query.
Verification and printing are outside each solver's timer. The first-puzzle BC
candidate checks use a separate KB and do not warm the timed solver's KB.

Our FC wrapper invokes the supplied pl_fc_entails separately for successive
cell/value queries. Although it indexes rules by premise, each library call
initializes counters, agenda and inferred state again. BC retains facts, proven,
expanded, waiting and remaining records across queries within a full-grid solve.
Its current implementation does not compare clause snapshots or invalidate a
cache after changes. Thus any observed timing advantage includes caching,
indexing and repeated-work differences, not just the inference direction.

The saved outputs above are measurements of this current-assignment version,
not results copied from another branch. Each puzzle/method has one timed sample;
runtimes depend on the environment and do not establish that BC is always faster.

In this run, the first puzzle took **15.12 s with FC** and **0.76 s with BC**,
as printed above. BC was faster for this implementation and workload; the
repeated-state initialization in FC and cross-query proof reuse in BC are
important reasons, so these numbers are not an algorithm-independent ranking.


## Part C: Streamlit Integration

Wrap your Sudoku solver and inference logic into an interactive Streamlit application, `sudoku_app.py`. Your application must implement the following:

**1. Puzzle selection & visual board display**
- An interactive selector/dropdown to pick any puzzle from `puzzles.json`.
- A visual rendering of the grid that clearly distinguishes the initial *givens* from empty cells.

**2. Full-grid auto-solver, with algorithm selection**
- A control (e.g. radio buttons) letting the user choose forward chaining (`solve_full_grid_fc`) or backward chaining (`solve_full_grid_bc`) before solving.
- A button that solves the full grid with the chosen algorithm, renders the solved state, and displays how long the solve took -- so the timing gap from task (d) is visible in the app, not just in the notebook.

**3. Targeted cell entailment query**
- Inputs for row ($r$), column ($c$), and value ($v$).
- A button that checks whether $\mathit{Is}_{rcv}$ is entailed (using `pl_bc_entails`) and displays the boolean verdict (`True` / `False`).

**4. Reasoning trace ("tutor mode")**
- Instrument your chosen inference algorithm (forward or backward chaining) to record the reasoning steps it takes while answering a query.
- Present that trace in a human-readable format -- not a raw Python string or internal symbol dictionary. For example: expandable cards/accordions showing the rule-firing sequence (*"Inferred $\mathit{Not}_{1,2,3}$ because row 1 already contains value 3"* $\implies$ *"Deduce $\mathit{Is}_{1,2,4}$ as the last remaining candidate"*), or plain-English sentences explaining each elimination by row, column, or box constraint.

Import `atom`, `build_definite_kb`, `build_general_kb`, `solve_full_grid_fc`, `solve_full_grid_bc`, and `pl_bc_entails` from `sudoku_solver.py`. Do not copy or rewrite these core functions in the Streamlit file. You may add app-specific helper functions where needed for the interface or reasoning trace.

Refer to the provided `StreamlitDeploymentGuide.pdf` guide to deploy your code and include the link for your Streamlit app below.


## Submission

Submit only the following three files:

1. `Sudoku_Assignment.ipynb` — with all required cells run, outputs visible, and all conceptual questions answered.
2. `sudoku_solver.py` — containing your implementation of the knowledge-base and inference functions.
3. `sudoku_app.py` — containing your Streamlit application.

The provided support files are required to run the assignment but are not part of the student submission.

### Deployed Streamlit app URL

https://daphnezhang627-cell-it5005-group7-sudoku-sudoku-app-curr-b0ge8r.streamlit.app/

Repository: `daphnezhang627-cell/it5005-group7-sudoku`  
Branch: `current-assignment`  
Entrypoint: `sudoku_app.py`  
Python: 3.12

Public-browser tests passed for all five supplied puzzles: complete FC and BC
grids matched the reference solutions, correct and incorrect candidate queries
returned the expected verdicts, and the recorded reasoning traces expanded
successfully. The original solver and app files are unchanged.

The app hides results when a different puzzle is selected; returning to the last
solved or queried puzzle can restore its previous result. This existing behavior
has been retained, not described as clearing the session state.
